# Phase 6: Execution, Market Microstructure & Systems  
## Notebook 06_12 — Latency Budget Profiler

### Research question

How can we decompose, simulate, profile, stress-test, and govern the latency budget of an execution pipeline from market-data arrival to order acknowledgement?

This notebook follows:

```text
06_01_almgren_chriss_optimal_execution
06_02_dynamic_programming_execution
06_03_square_root_market_impact_model
06_04_hawkes_process_order_flow
06_05_rough_volatility_estimation
06_06_microstructure_friction_cpp_core
06_07_limit_board_margin_and_deadband_rebalancing
06_08_avellaneda_stoikov_market_making
06_09_limit_order_fill_probability_model
06_10_smart_order_router_simulator
06_11_cpp_python_bindings_pybind11
06_12_latency_budget_profiler
```

The previous notebook built a C++/Python binding layer. This notebook asks whether the system is fast and stable enough to use those kernels in an execution pipeline.

It covers:

1. latency budget design;
2. event-time versus wall-clock latency;
3. pipeline stage definitions;
4. market-data ingest latency;
5. feature computation latency;
6. model inference latency;
7. risk-check latency;
8. order construction latency;
9. smart-order-router latency;
10. serialization latency;
11. network latency;
12. exchange acknowledgement latency;
13. fill-processing latency;
14. telemetry and persistence latency;
15. p50/p95/p99 latency;
16. tail latency and jitter;
17. SLA breach diagnostics;
18. bottleneck attribution;
19. microburst stress tests;
20. Python versus C++ boundary overhead;
21. batch versus event-by-event processing;
22. queueing/backpressure simulation;
23. kill-switch latency controls;
24. governance flags;
25. audit checks;
26. saved outputs and manifest.

The key idea:

> A trading system is only as fast as its slowest tail: p99 latency, not average latency, is what usually breaks execution quality.

## 1. Why latency budgeting matters

Execution systems are pipelines.

A typical path is:

```text
market data packet
→ parser
→ normaliser
→ feature engine
→ signal/model
→ risk checks
→ order construction
→ smart order router
→ serialization
→ network send
→ venue/broker acknowledgement
→ fill handling
→ persistence/telemetry
```

Each stage consumes time.

A latency budget assigns a target to every stage:

$$
TotalLatency = \sum_i StageLatency_i
$$

But averages are misleading. What matters is:

- p50: typical latency;
- p95: elevated latency;
- p99: tail latency;
- max: pathological latency;
- breach rate: frequency of SLA violation.

A strategy can tolerate a slow average less often than it can tolerate rare but severe tail spikes.

## 2. Latency vocabulary

### One-way latency

Time from signal decision to order reaching destination.

### Round-trip latency

Time from order send to acknowledgement or response.

### Tick-to-trade latency

Time from market-data event arrival to order send.

### Decision latency

Feature + model + risk + sizing + routing.

### Tail latency

High-percentile latency such as p99 or p99.9.

### Jitter

Variability in latency.

### Backpressure

When incoming events arrive faster than the system can process them.

### SLA

Service-level agreement or internal latency target.

## 3. Latency budget philosophy

A good latency budget should be:

1. **Stage-specific** — every component has a target.
2. **Percentile-based** — p95 and p99 matter.
3. **Stress-tested** — microbursts and volatility spikes matter.
4. **Actionable** — identify bottlenecks.
5. **Auditable** — save metrics and breach examples.
6. **Governed** — kill switches should trigger when latency invalidates execution assumptions.

This notebook is synthetic, but the structure mirrors how a real execution stack should be profiled.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class LatencyBudgetConfig:
    n_events: int = 120_000
    seed: int = 42
    base_event_rate_per_second: float = 750.0
    microburst_event_rate_per_second: float = 8_000.0
    microburst_probability: float = 0.006
    microburst_min_events: int = 80
    microburst_max_events: int = 600
    total_tick_to_trade_budget_us: float = 1_500.0
    total_ack_budget_us: float = 3_500.0
    p99_stage_breach_multiplier: float = 1.50
    queue_capacity_events: int = 20_000
    kill_switch_breach_rate_threshold: float = 0.02
    kill_switch_p99_latency_us: float = 4_000.0
    benchmark_repeats: int = 5
    output_subdir: str = "latency_budget_profiler_v1"

config = LatencyBudgetConfig()
config

## 5. Define latency budget by stage

We separate the pipeline into two parts:

### Tick-to-trade path

Market-data arrival to order send:

- market data ingest;
- parsing;
- normalisation;
- feature computation;
- model inference;
- risk checks;
- order construction;
- smart order routing;
- serialization.

### Order-response path

Order send to venue/broker response:

- network outbound;
- broker/venue gateway;
- exchange matching/ack;
- network inbound;
- acknowledgement handling;
- fill processing;
- telemetry/persistence.

Each stage has a target budget in microseconds.

In [ ]:
def make_stage_budget():
    stages = pd.DataFrame([
        {"stage": "market_data_ingest", "path": "tick_to_trade", "budget_us": 80.0, "base_us": 35.0, "jitter_us": 10.0, "tail_prob": 0.004, "tail_scale_us": 100.0},
        {"stage": "packet_parse", "path": "tick_to_trade", "budget_us": 55.0, "base_us": 25.0, "jitter_us": 8.0, "tail_prob": 0.003, "tail_scale_us": 80.0},
        {"stage": "normalise_book", "path": "tick_to_trade", "budget_us": 70.0, "base_us": 32.0, "jitter_us": 10.0, "tail_prob": 0.004, "tail_scale_us": 90.0},
        {"stage": "feature_engine", "path": "tick_to_trade", "budget_us": 220.0, "base_us": 120.0, "jitter_us": 45.0, "tail_prob": 0.010, "tail_scale_us": 220.0},
        {"stage": "model_inference", "path": "tick_to_trade", "budget_us": 260.0, "base_us": 135.0, "jitter_us": 55.0, "tail_prob": 0.012, "tail_scale_us": 260.0},
        {"stage": "risk_checks", "path": "tick_to_trade", "budget_us": 180.0, "base_us": 85.0, "jitter_us": 28.0, "tail_prob": 0.006, "tail_scale_us": 150.0},
        {"stage": "order_construction", "path": "tick_to_trade", "budget_us": 90.0, "base_us": 38.0, "jitter_us": 12.0, "tail_prob": 0.003, "tail_scale_us": 90.0},
        {"stage": "smart_order_router", "path": "tick_to_trade", "budget_us": 260.0, "base_us": 125.0, "jitter_us": 50.0, "tail_prob": 0.012, "tail_scale_us": 260.0},
        {"stage": "serialization", "path": "tick_to_trade", "budget_us": 95.0, "base_us": 44.0, "jitter_us": 14.0, "tail_prob": 0.004, "tail_scale_us": 100.0},

        {"stage": "network_outbound", "path": "ack_path", "budget_us": 450.0, "base_us": 190.0, "jitter_us": 80.0, "tail_prob": 0.010, "tail_scale_us": 450.0},
        {"stage": "broker_gateway", "path": "ack_path", "budget_us": 650.0, "base_us": 250.0, "jitter_us": 120.0, "tail_prob": 0.015, "tail_scale_us": 700.0},
        {"stage": "exchange_ack", "path": "ack_path", "budget_us": 850.0, "base_us": 310.0, "jitter_us": 160.0, "tail_prob": 0.020, "tail_scale_us": 900.0},
        {"stage": "network_inbound", "path": "ack_path", "budget_us": 450.0, "base_us": 185.0, "jitter_us": 75.0, "tail_prob": 0.010, "tail_scale_us": 430.0},
        {"stage": "ack_handler", "path": "ack_path", "budget_us": 140.0, "base_us": 65.0, "jitter_us": 20.0, "tail_prob": 0.005, "tail_scale_us": 140.0},
        {"stage": "fill_processing", "path": "ack_path", "budget_us": 240.0, "base_us": 105.0, "jitter_us": 45.0, "tail_prob": 0.010, "tail_scale_us": 240.0},
        {"stage": "telemetry_persistence", "path": "ack_path", "budget_us": 280.0, "base_us": 115.0, "jitter_us": 70.0, "tail_prob": 0.025, "tail_scale_us": 420.0},
    ])
    return stages

stage_budget = make_stage_budget()

stage_budget

## 6. Simulate market event arrivals and microbursts

Market-data arrivals are not uniform. They cluster during:

- opening auction;
- macro news;
- volatility bursts;
- quote storms;
- liquidation cascades.

We simulate baseline event arrivals with occasional microbursts.

In [ ]:
def simulate_event_arrivals(config):
    rng = np.random.default_rng(config.seed)

    event_ids = np.arange(config.n_events)
    interarrival_us = np.empty(config.n_events)

    t = 0
    microburst = np.zeros(config.n_events, dtype=int)

    while t < config.n_events:
        if rng.uniform() < config.microburst_probability:
            length = int(rng.integers(config.microburst_min_events, config.microburst_max_events + 1))
            end = min(config.n_events, t + length)
            rate = config.microburst_event_rate_per_second
            interarrival_us[t:end] = rng.exponential(1_000_000.0 / rate, size=end - t)
            microburst[t:end] = 1
            t = end
        else:
            interarrival_us[t] = rng.exponential(1_000_000.0 / config.base_event_rate_per_second)
            t += 1

    arrival_time_us = np.cumsum(interarrival_us)

    events = pd.DataFrame({
        "event_id": event_ids,
        "interarrival_us": interarrival_us,
        "arrival_time_us": arrival_time_us,
        "microburst": microburst,
    })

    return events

events = simulate_event_arrivals(config)

events.head(), events["microburst"].mean()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(events["event_id"].head(5000), events["interarrival_us"].head(5000))
plt.title("Interarrival Times — First 5,000 Events")
plt.xlabel("Event")
plt.ylabel("Interarrival, us")
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(events["interarrival_us"].clip(upper=5000), bins=80)
plt.title("Interarrival Distribution")
plt.xlabel("Interarrival, us")
plt.ylabel("Count")
plt.show()

## 7. Simulate per-stage latency

Each stage latency is:

$$
\begin{aligned}
latency &= base \\
&\quad + jitter \\
&\quad + tailShock \\
&\quad + microburstPenalty
\end{aligned}
$$

The tail shock creates rare p99-type events.

The microburst penalty increases processing latency during load spikes.

In [ ]:
def simulate_stage_latencies(events, stage_budget, config):
    rng = np.random.default_rng(config.seed + 101)
    n = len(events)

    rows = []

    for _, stage in stage_budget.iterrows():
        base = stage["base_us"]
        jitter = np.abs(rng.normal(0.0, stage["jitter_us"], size=n))
        tail = rng.exponential(stage["tail_scale_us"], size=n) * (rng.uniform(size=n) < stage["tail_prob"])
        microburst_penalty = events["microburst"].to_numpy() * rng.exponential(stage["jitter_us"] * 2.5, size=n)

        latency = base + jitter + tail + microburst_penalty
        latency = np.maximum(latency, 0.0)

        rows.append(pd.DataFrame({
            "event_id": events["event_id"].to_numpy(),
            "stage": stage["stage"],
            "path": stage["path"],
            "budget_us": stage["budget_us"],
            "latency_us": latency,
            "microburst": events["microburst"].to_numpy(),
        }))

    stage_latencies = pd.concat(rows, ignore_index=True)
    return stage_latencies

stage_latencies = simulate_stage_latencies(events, stage_budget, config)

stage_latencies.head()

## 8. Stage-level latency profile

Compute:

- mean;
- p50;
- p95;
- p99;
- max;
- standard deviation;
- budget breach rate.

In [ ]:
def percentile(x, q):
    return float(np.percentile(x, q))

def stage_latency_profile(stage_latencies):
    profile = (
        stage_latencies
        .groupby(["stage", "path", "budget_us"])
        .agg(
            n=("latency_us", "count"),
            mean_us=("latency_us", "mean"),
            std_us=("latency_us", "std"),
            p50_us=("latency_us", lambda x: percentile(x, 50)),
            p95_us=("latency_us", lambda x: percentile(x, 95)),
            p99_us=("latency_us", lambda x: percentile(x, 99)),
            max_us=("latency_us", "max"),
            breach_rate=("latency_us", lambda x: np.mean(x > x.name if False else 0.0)),
        )
        .reset_index()
    )

    # Compute breach rate with budget.
    breach_rates = (
        stage_latencies
        .assign(breach=lambda df: df["latency_us"] > df["budget_us"])
        .groupby("stage")["breach"]
        .mean()
        .rename("breach_rate")
        .reset_index()
    )

    profile = profile.drop(columns=["breach_rate"]).merge(breach_rates, on="stage", how="left")
    profile["p99_budget_ratio"] = profile["p99_us"] / profile["budget_us"]
    profile["mean_budget_ratio"] = profile["mean_us"] / profile["budget_us"]

    return profile.sort_values("p99_budget_ratio", ascending=False)

stage_profile = stage_latency_profile(stage_latencies)

stage_profile

In [ ]:
plt.figure(figsize=(12, 7))
plot_df = stage_profile.sort_values("p99_us")
plt.barh(plot_df["stage"], plot_df["p99_us"], label="p99")
plt.scatter(plot_df["budget_us"], plot_df["stage"], label="budget", marker="x")
plt.title("Stage p99 Latency vs Budget")
plt.xlabel("Latency, us")
plt.ylabel("Stage")
plt.legend()
plt.show()

## 9. End-to-end latency reconstruction

For each event:

$$
tickToTradeLatency = \sum_{i\in tickToTrade} latency_i
$$

$$
ackLatency = \sum_{i\in ackPath} latency_i
$$

$$
totalLatency = tickToTradeLatency + ackLatency
$$

In [ ]:
def reconstruct_end_to_end_latency(stage_latencies):
    pivot = stage_latencies.pivot_table(
        index="event_id",
        columns="stage",
        values="latency_us",
        aggfunc="sum",
    ).fillna(0.0)

    stage_path = stage_latencies[["stage", "path"]].drop_duplicates().set_index("stage")["path"]
    tick_stages = stage_path[stage_path == "tick_to_trade"].index.tolist()
    ack_stages = stage_path[stage_path == "ack_path"].index.tolist()

    e2e = pd.DataFrame(index=pivot.index)
    e2e["tick_to_trade_us"] = pivot[tick_stages].sum(axis=1)
    e2e["ack_path_us"] = pivot[ack_stages].sum(axis=1)
    e2e["total_order_lifecycle_us"] = e2e["tick_to_trade_us"] + e2e["ack_path_us"]

    e2e = e2e.join(events.set_index("event_id")[["arrival_time_us", "interarrival_us", "microburst"]], how="left")

    e2e["tick_to_trade_breach"] = e2e["tick_to_trade_us"] > config.total_tick_to_trade_budget_us
    e2e["ack_breach"] = e2e["ack_path_us"] > config.total_ack_budget_us
    e2e["lifecycle_breach"] = e2e["tick_to_trade_breach"] | e2e["ack_breach"]

    return e2e, pivot

e2e_latency, latency_matrix = reconstruct_end_to_end_latency(stage_latencies)

e2e_latency.head()

## 10. End-to-end latency profile

In [ ]:
def latency_distribution_summary(e2e_latency):
    rows = []

    for col in ["tick_to_trade_us", "ack_path_us", "total_order_lifecycle_us"]:
        x = e2e_latency[col]
        rows.append({
            "metric": col,
            "mean_us": x.mean(),
            "std_us": x.std(),
            "p50_us": x.quantile(0.50),
            "p95_us": x.quantile(0.95),
            "p99_us": x.quantile(0.99),
            "p999_us": x.quantile(0.999),
            "max_us": x.max(),
        })

    summary = pd.DataFrame(rows)

    breach_summary = pd.DataFrame([{
        "tick_to_trade_breach_rate": e2e_latency["tick_to_trade_breach"].mean(),
        "ack_breach_rate": e2e_latency["ack_breach"].mean(),
        "lifecycle_breach_rate": e2e_latency["lifecycle_breach"].mean(),
        "microburst_event_rate": e2e_latency["microburst"].mean(),
    }])

    return summary, breach_summary

e2e_summary, breach_summary = latency_distribution_summary(e2e_latency)

e2e_summary, breach_summary

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(e2e_latency["tick_to_trade_us"], bins=100, alpha=0.65, label="tick-to-trade")
plt.axvline(config.total_tick_to_trade_budget_us, linestyle="--", label="tick-to-trade budget")
plt.title("Tick-to-Trade Latency Distribution")
plt.xlabel("Latency, us")
plt.ylabel("Count")
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(e2e_latency["ack_path_us"], bins=100, alpha=0.65, label="ack path")
plt.axvline(config.total_ack_budget_us, linestyle="--", label="ack budget")
plt.title("Ack Path Latency Distribution")
plt.xlabel("Latency, us")
plt.ylabel("Count")
plt.legend()
plt.show()

## 11. Bottleneck attribution

There are two common bottleneck views:

### Average contribution

Which stages consume the most total latency on average?

### Tail contribution

Which stages dominate during p99 end-to-end latency events?

The second view is usually more important.

In [ ]:
def bottleneck_attribution(stage_latencies, e2e_latency):
    mean_stage = (
        stage_latencies
        .groupby("stage")
        .agg(mean_latency_us=("latency_us", "mean"))
        .reset_index()
    )
    mean_stage["mean_contribution_share"] = mean_stage["mean_latency_us"] / mean_stage["mean_latency_us"].sum()

    p99_threshold = e2e_latency["total_order_lifecycle_us"].quantile(0.99)
    tail_events = e2e_latency[e2e_latency["total_order_lifecycle_us"] >= p99_threshold].index

    tail_stage = (
        stage_latencies[stage_latencies["event_id"].isin(tail_events)]
        .groupby("stage")
        .agg(tail_mean_latency_us=("latency_us", "mean"))
        .reset_index()
    )
    tail_stage["tail_contribution_share"] = tail_stage["tail_mean_latency_us"] / tail_stage["tail_mean_latency_us"].sum()

    out = mean_stage.merge(tail_stage, on="stage", how="outer").fillna(0.0)
    out["tail_minus_mean_share"] = out["tail_contribution_share"] - out["mean_contribution_share"]

    return out.sort_values("tail_contribution_share", ascending=False)

bottleneck_report = bottleneck_attribution(stage_latencies, e2e_latency)

bottleneck_report

In [ ]:
plt.figure(figsize=(12, 7))
plot_df = bottleneck_report.sort_values("tail_contribution_share")
plt.barh(plot_df["stage"], plot_df["mean_contribution_share"], alpha=0.6, label="mean share")
plt.barh(plot_df["stage"], plot_df["tail_contribution_share"], alpha=0.6, label="p99 tail share")
plt.title("Mean vs Tail Bottleneck Attribution")
plt.xlabel("Contribution share")
plt.ylabel("Stage")
plt.legend()
plt.show()

## 12. Microburst impact

Compare latency during normal periods and microbursts.

In [ ]:
microburst_report = (
    e2e_latency
    .groupby("microburst")
    .agg(
        n=("total_order_lifecycle_us", "count"),
        mean_tick_to_trade_us=("tick_to_trade_us", "mean"),
        p99_tick_to_trade_us=("tick_to_trade_us", lambda x: x.quantile(0.99)),
        mean_ack_us=("ack_path_us", "mean"),
        p99_ack_us=("ack_path_us", lambda x: x.quantile(0.99)),
        lifecycle_breach_rate=("lifecycle_breach", "mean"),
    )
    .reset_index()
)

microburst_report["regime"] = np.where(microburst_report["microburst"] == 1, "microburst", "normal")

microburst_report

## 13. Queueing and backpressure simulation

If events arrive faster than they are processed, a queue builds.

For each event:

$$
start_i = \max(arrival_i, finish_{i-1})
$$

$$
finish_i = start_i + processingLatency_i
$$

$$
queueDelay_i = start_i - arrival_i
$$

This is a simple single-server approximation.

In [ ]:
def simulate_queueing(e2e_latency, config):
    arr = e2e_latency["arrival_time_us"].to_numpy()
    service = e2e_latency["tick_to_trade_us"].to_numpy()

    n = len(arr)
    start = np.empty(n)
    finish = np.empty(n)
    queue_delay = np.empty(n)
    queue_depth_proxy = np.empty(n)

    prev_finish = 0.0
    for i in range(n):
        start[i] = max(arr[i], prev_finish)
        finish[i] = start[i] + service[i]
        queue_delay[i] = start[i] - arr[i]
        prev_finish = finish[i]

    # Approximate queue depth by counting events that arrived before current finish but not yet processed.
    # Efficient rolling approximation using arrival index and finish times.
    j = 0
    for i in range(n):
        while j < n and arr[j] <= finish[i]:
            j += 1
        queue_depth_proxy[i] = max(j - i - 1, 0)

    q = pd.DataFrame({
        "event_id": e2e_latency.index,
        "arrival_time_us": arr,
        "service_latency_us": service,
        "processing_start_us": start,
        "processing_finish_us": finish,
        "queue_delay_us": queue_delay,
        "queue_depth_proxy": queue_depth_proxy,
        "microburst": e2e_latency["microburst"].to_numpy(),
    }).set_index("event_id")

    q["queue_capacity_breach"] = q["queue_depth_proxy"] > config.queue_capacity_events
    q["queue_delay_breach"] = q["queue_delay_us"] > config.total_tick_to_trade_budget_us

    return q

queue_report = simulate_queueing(e2e_latency, config)

queue_report.head()

In [ ]:
queue_summary = pd.DataFrame([{
    "mean_queue_delay_us": queue_report["queue_delay_us"].mean(),
    "p95_queue_delay_us": queue_report["queue_delay_us"].quantile(0.95),
    "p99_queue_delay_us": queue_report["queue_delay_us"].quantile(0.99),
    "max_queue_delay_us": queue_report["queue_delay_us"].max(),
    "mean_queue_depth": queue_report["queue_depth_proxy"].mean(),
    "p99_queue_depth": queue_report["queue_depth_proxy"].quantile(0.99),
    "max_queue_depth": queue_report["queue_depth_proxy"].max(),
    "queue_capacity_breach_rate": queue_report["queue_capacity_breach"].mean(),
    "queue_delay_breach_rate": queue_report["queue_delay_breach"].mean(),
}])

queue_summary

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(queue_report.index[:10000], queue_report["queue_depth_proxy"].iloc[:10000])
plt.title("Queue Depth Proxy — First 10,000 Events")
plt.xlabel("Event")
plt.ylabel("Queue depth proxy")
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(queue_report["queue_delay_us"].clip(upper=10_000), bins=100)
plt.title("Queue Delay Distribution")
plt.xlabel("Queue delay, us")
plt.ylabel("Count")
plt.show()

## 14. Stage breach examples

Save examples of the worst events for audit and debugging.

In [ ]:
worst_events = e2e_latency.sort_values("total_order_lifecycle_us", ascending=False).head(20)

worst_event_stage_detail = (
    stage_latencies[stage_latencies["event_id"].isin(worst_events.index)]
    .pivot_table(index="event_id", columns="stage", values="latency_us", aggfunc="sum")
    .join(e2e_latency[["tick_to_trade_us", "ack_path_us", "total_order_lifecycle_us", "microburst"]])
    .sort_values("total_order_lifecycle_us", ascending=False)
)

worst_event_stage_detail.head()

## 15. Python versus vectorised latency microbenchmark

This is a small local microbenchmark showing why hot loops should avoid Python event-by-event processing where possible.

We compare:

1. Python loop feature computation;
2. NumPy vectorised feature computation.

This is not a replacement for production profiling, but it demonstrates the direction.

In [ ]:
def python_loop_feature(x, window):
    out = np.full(len(x), np.nan)
    for i in range(window - 1, len(x)):
        out[i] = np.mean(np.abs(np.diff(x[i-window+1:i+1])))
    return out

def numpy_vectorized_feature(x, window):
    dx = np.abs(np.diff(x, prepend=x[0]))
    kernel = np.ones(window) / window
    out = np.convolve(dx, kernel, mode="full")[:len(dx)]
    out[:window-1] = np.nan
    return out

def microbenchmark_features(config):
    rng = np.random.default_rng(config.seed + 777)
    x = np.cumsum(rng.normal(0, 1, 50_000))
    window = 64
    rows = []

    for repeat in range(config.benchmark_repeats):
        start = time.perf_counter()
        loop_out = python_loop_feature(x, window)
        elapsed = time.perf_counter() - start
        rows.append({
            "method": "python_loop",
            "repeat": repeat,
            "elapsed_seconds": elapsed,
            "items_per_second": len(x) / elapsed,
        })

    for repeat in range(config.benchmark_repeats):
        start = time.perf_counter()
        vec_out = numpy_vectorized_feature(x, window)
        elapsed = time.perf_counter() - start
        rows.append({
            "method": "numpy_vectorized",
            "repeat": repeat,
            "elapsed_seconds": elapsed,
            "items_per_second": len(x) / elapsed,
        })

    diff = np.nanmax(np.abs(loop_out - vec_out))

    return pd.DataFrame(rows), diff

feature_benchmark, feature_diff = microbenchmark_features(config)

feature_benchmark_summary = (
    feature_benchmark
    .groupby("method")
    .agg(
        mean_elapsed_seconds=("elapsed_seconds", "mean"),
        mean_items_per_second=("items_per_second", "mean"),
    )
    .reset_index()
)

feature_benchmark_summary, feature_diff

## 16. Batch versus event processing

Batching can improve throughput but adds waiting time.

A batch of size $B$ creates average waiting delay roughly:

$$
wait \approx \frac{B-1}{2\lambda}
$$

where $\lambda$ is event rate.

This trade-off matters:

- event-by-event processing minimises waiting but may be expensive;
- batching improves compute efficiency but adds latency.

In [ ]:
def batch_latency_tradeoff(events, base_compute_us_per_event=140.0):
    rows = []

    for batch_size in [1, 2, 4, 8, 16, 32, 64, 128]:
        # compute efficiency improves with batch size but has fixed overhead.
        fixed_overhead_us = 60.0
        compute_per_event = base_compute_us_per_event / (1.0 + 0.25 * np.log2(batch_size))
        compute_latency_us = fixed_overhead_us + compute_per_event * batch_size

        # Estimate waiting delay from empirical interarrival.
        mean_interarrival = events["interarrival_us"].mean()
        avg_wait_us = (batch_size - 1) * mean_interarrival / 2.0
        p95_wait_us = (batch_size - 1) * events["interarrival_us"].quantile(0.95) / 2.0

        per_event_compute = compute_latency_us / batch_size
        total_mean_per_event = per_event_compute + avg_wait_us

        rows.append({
            "batch_size": batch_size,
            "compute_latency_batch_us": compute_latency_us,
            "compute_latency_per_event_us": per_event_compute,
            "avg_wait_us": avg_wait_us,
            "p95_wait_us": p95_wait_us,
            "total_mean_per_event_us": total_mean_per_event,
        })

    return pd.DataFrame(rows)

batch_tradeoff = batch_latency_tradeoff(events)

batch_tradeoff

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(batch_tradeoff["batch_size"], batch_tradeoff["compute_latency_per_event_us"], marker="o", label="compute per event")
plt.plot(batch_tradeoff["batch_size"], batch_tradeoff["avg_wait_us"], marker="o", label="average wait")
plt.plot(batch_tradeoff["batch_size"], batch_tradeoff["total_mean_per_event_us"], marker="o", label="total mean")
plt.xscale("log", base=2)
plt.title("Batch Size Latency Trade-off")
plt.xlabel("Batch size")
plt.ylabel("Latency, us")
plt.legend()
plt.show()

## 17. Tail-latency stress scenario

Stress scenario:

- network tails double;
- model inference tails increase;
- telemetry persistence slows down;
- microbursts are more frequent.

We compare base versus stress.

In [ ]:
def make_stress_config(config):
    return LatencyBudgetConfig(
        n_events=config.n_events,
        seed=config.seed + 2024,
        base_event_rate_per_second=config.base_event_rate_per_second,
        microburst_event_rate_per_second=config.microburst_event_rate_per_second * 1.4,
        microburst_probability=config.microburst_probability * 2.5,
        microburst_min_events=config.microburst_min_events,
        microburst_max_events=config.microburst_max_events * 2,
        total_tick_to_trade_budget_us=config.total_tick_to_trade_budget_us,
        total_ack_budget_us=config.total_ack_budget_us,
        p99_stage_breach_multiplier=config.p99_stage_breach_multiplier,
        queue_capacity_events=config.queue_capacity_events,
        kill_switch_breach_rate_threshold=config.kill_switch_breach_rate_threshold,
        kill_switch_p99_latency_us=config.kill_switch_p99_latency_us,
        benchmark_repeats=config.benchmark_repeats,
        output_subdir=config.output_subdir,
    )

def stress_stage_budget(stage_budget):
    stressed = stage_budget.copy()
    stress_stages = ["model_inference", "smart_order_router", "network_outbound", "broker_gateway", "exchange_ack", "network_inbound", "telemetry_persistence"]
    mask = stressed["stage"].isin(stress_stages)
    stressed.loc[mask, "tail_prob"] *= 2.0
    stressed.loc[mask, "tail_scale_us"] *= 1.8
    stressed.loc[stressed["stage"] == "telemetry_persistence", "base_us"] *= 1.5
    return stressed

stress_config = make_stress_config(config)
stress_events = simulate_event_arrivals(stress_config)
stress_budget = stress_stage_budget(stage_budget)
stress_latencies = simulate_stage_latencies(stress_events, stress_budget, stress_config)
stress_e2e, stress_matrix = reconstruct_end_to_end_latency(stress_latencies)
stress_summary, stress_breach = latency_distribution_summary(stress_e2e)

base_vs_stress = pd.concat([
    e2e_summary.assign(scenario="base"),
    stress_summary.assign(scenario="stress"),
], ignore_index=True)

base_vs_stress_breach = pd.concat([
    breach_summary.assign(scenario="base"),
    stress_breach.assign(scenario="stress"),
], ignore_index=True)

base_vs_stress, base_vs_stress_breach

## 18. Kill-switch latency controls

Latency can invalidate execution assumptions.

Example controls:

1. block new passive orders if p99 tick-to-trade latency is too high;
2. force marketable-only routing during queue backlog;
3. stop trading if lifecycle breach rate exceeds threshold;
4. shed non-critical telemetry during overload;
5. reduce model complexity under stress.

We implement a simple latency control decision.

In [ ]:
def latency_kill_switch_decision(e2e_latency, queue_report, config):
    recent = e2e_latency.tail(max(1, len(e2e_latency) // 10))
    recent_queue = queue_report.tail(max(1, len(queue_report) // 10))

    recent_p99_tick = recent["tick_to_trade_us"].quantile(0.99)
    recent_breach_rate = recent["lifecycle_breach"].mean()
    recent_queue_p99 = recent_queue["queue_depth_proxy"].quantile(0.99)

    block_passive = recent_p99_tick > config.kill_switch_p99_latency_us
    marketable_only = recent_queue_p99 > config.queue_capacity_events * 0.50
    stop_trading = recent_breach_rate > config.kill_switch_breach_rate_threshold

    if stop_trading:
        status = "RED_STOP_TRADING"
    elif block_passive or marketable_only:
        status = "AMBER_REDUCE_COMPLEXITY"
    else:
        status = "GREEN_NORMAL"

    return pd.DataFrame([{
        "recent_p99_tick_to_trade_us": recent_p99_tick,
        "recent_lifecycle_breach_rate": recent_breach_rate,
        "recent_p99_queue_depth": recent_queue_p99,
        "block_passive_orders": bool(block_passive),
        "marketable_only_mode": bool(marketable_only),
        "stop_trading": bool(stop_trading),
        "latency_control_status": status,
    }])

latency_control = latency_kill_switch_decision(e2e_latency, queue_report, config)

latency_control

## 19. Latency budget recommendations

Create an actionable recommendation table:

- stages with p99 above budget;
- stages with high breach rate;
- stages dominating tail latency;
- suggested remediation.

In [ ]:
def remediation_recommendation(stage):
    mapping = {
        "market_data_ingest": "Check feed handler, kernel bypass, NIC settings and packet drops.",
        "packet_parse": "Optimise parser, avoid allocations, use fixed schema.",
        "normalise_book": "Move book updates to C++/Rust, reduce object churn.",
        "feature_engine": "Precompute features, vectorise, move hot path to C++.",
        "model_inference": "Use smaller model, cache features, compile inference path.",
        "risk_checks": "Precompute limits, use branch-light checks, avoid database calls.",
        "order_construction": "Use object pools and fixed-size messages.",
        "smart_order_router": "Reduce venue search, precompute scores, move router core to C++.",
        "serialization": "Use binary protocol and preallocated buffers.",
        "network_outbound": "Check routing path, colocated gateway, NIC and kernel settings.",
        "broker_gateway": "Escalate gateway latency, monitor broker SLA.",
        "exchange_ack": "Venue condition; monitor exchange/broker round-trip separately.",
        "network_inbound": "Check network path, packet loss and timestamping.",
        "ack_handler": "Avoid locks and heavy logging in acknowledgement path.",
        "fill_processing": "Move fill accounting to compiled kernel, reduce synchronous work.",
        "telemetry_persistence": "Make telemetry asynchronous and shed under stress.",
    }
    return mapping.get(stage, "Profile stage and inspect tail-latency causes.")

recommendation_table = stage_profile.copy()
recommendation_table["needs_attention"] = (
    (recommendation_table["p99_us"] > recommendation_table["budget_us"])
    | (recommendation_table["breach_rate"] > 0.01)
    | (recommendation_table["p99_budget_ratio"] > config.p99_stage_breach_multiplier)
)
recommendation_table["recommendation"] = recommendation_table["stage"].map(remediation_recommendation)

recommendation_table = recommendation_table.sort_values(
    ["needs_attention", "p99_budget_ratio"],
    ascending=[False, False],
)

recommendation_table.head(20)

## 20. Governance flags

A latency profiler should be reviewed if:

1. tick-to-trade breach rate exceeds threshold;
2. ack breach rate exceeds threshold;
3. p99 tick-to-trade exceeds kill-switch threshold;
4. queue backpressure is severe;
5. microbursts create high breach rates;
6. any stage p99 exceeds its budget by too much;
7. telemetry is part of the critical path;
8. no real production timestamps are used.

In [ ]:
tick_breach = breach_summary["tick_to_trade_breach_rate"].iloc[0]
ack_breach = breach_summary["ack_breach_rate"].iloc[0]
lifecycle_breach = breach_summary["lifecycle_breach_rate"].iloc[0]
p99_tick = e2e_summary.loc[e2e_summary["metric"] == "tick_to_trade_us", "p99_us"].iloc[0]
p99_ack = e2e_summary.loc[e2e_summary["metric"] == "ack_path_us", "p99_us"].iloc[0]
queue_p99_depth = queue_summary["p99_queue_depth"].iloc[0]
max_stage_ratio = stage_profile["p99_budget_ratio"].max()
microburst_breach = microburst_report.loc[microburst_report["microburst"] == 1, "lifecycle_breach_rate"].iloc[0] if 1 in set(microburst_report["microburst"]) else 0.0
telemetry_tail_share = bottleneck_report.loc[bottleneck_report["stage"] == "telemetry_persistence", "tail_contribution_share"].iloc[0]

governance_flags = pd.DataFrame([{
    "tick_to_trade_breach_rate": tick_breach,
    "ack_breach_rate": ack_breach,
    "lifecycle_breach_rate": lifecycle_breach,
    "p99_tick_to_trade_us": p99_tick,
    "p99_ack_us": p99_ack,
    "p99_queue_depth": queue_p99_depth,
    "max_stage_p99_budget_ratio": max_stage_ratio,
    "microburst_lifecycle_breach_rate": microburst_breach,
    "telemetry_tail_contribution_share": telemetry_tail_share,
    "latency_control_status": latency_control["latency_control_status"].iloc[0],
    "flag_tick_breach_rate_high": bool(tick_breach > config.kill_switch_breach_rate_threshold),
    "flag_ack_breach_rate_high": bool(ack_breach > config.kill_switch_breach_rate_threshold),
    "flag_p99_tick_too_high": bool(p99_tick > config.kill_switch_p99_latency_us),
    "flag_queue_backpressure_high": bool(queue_p99_depth > config.queue_capacity_events * 0.50),
    "flag_stage_budget_overrun": bool(max_stage_ratio > config.p99_stage_breach_multiplier),
    "flag_microburst_breach_high": bool(microburst_breach > config.kill_switch_breach_rate_threshold * 2.0),
    "flag_telemetry_in_critical_tail": bool(telemetry_tail_share > 0.12),
    "flag_synthetic_not_real_timestamps": True,
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_tick_breach_rate_high",
        "flag_ack_breach_rate_high",
        "flag_p99_tick_too_high",
        "flag_queue_backpressure_high",
        "flag_stage_budget_overrun",
        "flag_microburst_breach_high",
        "flag_telemetry_in_critical_tail",
        "flag_synthetic_not_real_timestamps",
    ]
].any(axis=1)

governance_flags

## 21. Audit checks

Numerical and process checks:

1. event arrivals are increasing;
2. stage latencies are non-negative;
3. every stage has a budget;
4. end-to-end latencies are finite;
5. queue delay is non-negative;
6. benchmark outputs are finite;
7. governance flags exist.

In [ ]:
audit_rows = []

arrivals_increasing = bool(events["arrival_time_us"].is_monotonic_increasing)
audit_rows.append({
    "check": "arrival_times_increasing",
    "value": arrivals_increasing,
    "passed": arrivals_increasing,
})

latencies_nonnegative = bool((stage_latencies["latency_us"] >= 0).all())
audit_rows.append({
    "check": "stage_latencies_nonnegative",
    "value": latencies_nonnegative,
    "passed": latencies_nonnegative,
})

all_stages_budgeted = bool(stage_latencies["budget_us"].notna().all())
audit_rows.append({
    "check": "all_stages_have_budget",
    "value": all_stages_budgeted,
    "passed": all_stages_budgeted,
})

e2e_finite = bool(np.isfinite(e2e_latency[["tick_to_trade_us", "ack_path_us", "total_order_lifecycle_us"]].to_numpy()).all())
audit_rows.append({
    "check": "end_to_end_latencies_finite",
    "value": e2e_finite,
    "passed": e2e_finite,
})

queue_nonnegative = bool((queue_report["queue_delay_us"] >= -1e-12).all())
audit_rows.append({
    "check": "queue_delay_nonnegative",
    "value": queue_nonnegative,
    "passed": queue_nonnegative,
})

benchmark_finite = bool(np.isfinite(feature_benchmark[["elapsed_seconds", "items_per_second"]].to_numpy()).all())
audit_rows.append({
    "check": "feature_benchmark_outputs_finite",
    "value": benchmark_finite,
    "passed": benchmark_finite,
})

governance_exists = bool(len(governance_flags) == 1)
audit_rows.append({
    "check": "governance_flags_exist",
    "value": governance_exists,
    "passed": governance_exists,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 22. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed") / config.output_subdir
output_dir.mkdir(parents=True, exist_ok=True)

stage_budget.to_csv(output_dir / "stage_budget.csv", index=False)
events.to_csv(output_dir / "event_arrivals.csv", index=False)
stage_latencies.to_csv(output_dir / "stage_latencies.csv", index=False)
stage_profile.to_csv(output_dir / "stage_latency_profile.csv", index=False)
e2e_latency.to_csv(output_dir / "end_to_end_latency.csv")
latency_matrix.to_csv(output_dir / "latency_matrix.csv")
e2e_summary.to_csv(output_dir / "end_to_end_summary.csv", index=False)
breach_summary.to_csv(output_dir / "breach_summary.csv", index=False)
bottleneck_report.to_csv(output_dir / "bottleneck_report.csv", index=False)
microburst_report.to_csv(output_dir / "microburst_report.csv", index=False)
queue_report.to_csv(output_dir / "queue_report.csv")
queue_summary.to_csv(output_dir / "queue_summary.csv", index=False)
worst_event_stage_detail.to_csv(output_dir / "worst_event_stage_detail.csv")
feature_benchmark.to_csv(output_dir / "feature_benchmark.csv", index=False)
feature_benchmark_summary.to_csv(output_dir / "feature_benchmark_summary.csv", index=False)
batch_tradeoff.to_csv(output_dir / "batch_latency_tradeoff.csv", index=False)
base_vs_stress.to_csv(output_dir / "base_vs_stress_latency.csv", index=False)
base_vs_stress_breach.to_csv(output_dir / "base_vs_stress_breach.csv", index=False)
latency_control.to_csv(output_dir / "latency_control.csv", index=False)
recommendation_table.to_csv(output_dir / "recommendation_table.csv", index=False)
governance_flags.to_csv(output_dir / "governance_flags.csv", index=False)
audit_report.to_csv(output_dir / "audit_report.csv", index=False)

manifest = {
    "dataset_name": "latency_budget_profiler_outputs",
    "schema_version": "latency_budget_profiler_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "n_events": int(len(events)),
    "stage_count": int(stage_budget["stage"].nunique()),
    "end_to_end_summary": e2e_summary.to_dict(orient="records"),
    "breach_summary": breach_summary.to_dict(orient="records"),
    "queue_summary": queue_summary.to_dict(orient="records"),
    "latency_control": latency_control.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "This notebook simulates and profiles a latency budget for an execution pipeline.",
        "Latency is decomposed by stage and path: tick-to-trade and acknowledgement path.",
        "p50, p95, p99, p999 and breach rates are reported.",
        "Microbursts, queueing/backpressure and stress scenarios are included.",
        "Feature-computation microbenchmarks compare Python loop and vectorised approaches.",
        "Latency kill-switch decisions and governance flags are generated.",
        "The data is synthetic and should be replaced with real timestamped production telemetry."
    ],
}

manifest_path = output_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

output_dir / "stage_latency_profile.csv", output_dir / "end_to_end_summary.csv", output_dir / "governance_flags.csv", manifest_path

## 23. Practical checklist for latency profiling

Before trusting an execution system:

1. **Timestamp every stage.**
2. **Use monotonic clocks.**
3. **Separate tick-to-trade from ack latency.**
4. **Report p50/p95/p99, not only mean.**
5. **Profile under microbursts.**
6. **Measure queueing delay.**
7. **Keep telemetry off the critical path.**
8. **Set kill-switch thresholds.**
9. **Save worst-event traces.**
10. **Compare expected and actual latency budgets daily.**

## 24. Limitations

### Synthetic timestamps

This notebook simulates latency. Real latency profiling requires hardware/software timestamps.

### No OS/network stack detail

Kernel bypass, NIC queues, interrupt moderation, CPU pinning, garbage collection and NUMA are not modelled directly.

### Single-server queue

Backpressure is represented with a simple single-server queue.

### No real broker or exchange

Network and exchange acknowledgement latencies are synthetic.

### No distributed tracing

A production system should use trace IDs across services.

### No clock synchronisation

Real systems require clock synchronisation such as PTP or GPS-grade timestamping for high precision.

## 25. Research frontier and extensions

Important extensions include:

1. real production trace ingestion;
2. hardware timestamp integration;
3. distributed tracing with trace IDs;
4. CPU-core and NUMA profiling;
5. GC and allocation profiling;
6. kernel bypass / DPDK experiments;
7. adaptive complexity reduction under stress;
8. router-specific latency attribution;
9. live latency SLA dashboards;
10. latency-aware backtesting and execution simulation.

## 26. Suggested follow-up notebooks

This notebook naturally leads to:

1. **06_13_production_reconciliation_and_breaks**  
   Reconcile intended actions, sent orders, acknowledgements, fills and latency traces.

2. **Phase 7 execution capstone**  
   Combine latency, routing, fill probability, impact and risk controls into a full production-style execution research project.

## 27. Summary

This notebook implemented a latency budget profiler.

It showed how to:

1. define stage-level latency budgets;
2. simulate event arrivals and microbursts;
3. simulate per-stage latency;
4. compute stage p50/p95/p99 profiles;
5. reconstruct tick-to-trade and ack latency;
6. compute breach rates;
7. attribute mean and tail bottlenecks;
8. analyse microburst behaviour;
9. simulate queueing and backpressure;
10. save worst-event traces;
11. benchmark Python loop versus vectorised feature computation;
12. analyse batch-size latency trade-offs;
13. stress tail latency;
14. generate latency kill-switch decisions;
15. create governance flags and audit checks;
16. save outputs and manifests.

The key computational takeaway:

> Latency profiling is a percentile and bottleneck-attribution problem, not an average-timing exercise.

The key financial takeaway:

> A strategy with good alpha can still fail if latency tails make its fills stale, toxic, or unreachable.

## 28. Further reading

- High-performance trading-system engineering literature.
- Linux performance, CPU pinning, NUMA and kernel bypass references.
- DPDK and Solarflare/OpenOnload-style network acceleration materials.
- Distributed tracing and observability documentation.
- Exchange and broker connectivity specifications.
- Production incident postmortems on tail latency and backpressure.